# Makemore: Wavenet

Following Karpathy's *"Building makemore Part 5: Building a WaveNet"* (Zero to Hero, video 6). Two things happen in this video at once: (1) the codebase gets refactored into PyTorch-style modules (`Linear`, `BatchNorm1d`, `Tanh`, `Embedding`, `FlattenConsecutive`, `Sequential`) that mirror `torch.nn`, and (2) we introduce a WaveNet-inspired hierarchical architecture where consecutive character embeddings are fused gradually (pairs → pairs-of-pairs → ...) instead of flattening everything at the input. Context length bumps from 3 to 8 chars; with tuned hyperparameters (n_embed=24, n_hidden=128) the dev loss drops below 2.0 for the first time in the series.

In [75]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random
%matplotlib inline
plt.style.use('dark_background')

In [76]:
# Read in all the words
words = open('../data/names.txt', 'r').read().splitlines()
print(len(words))
print(max(len(word) for word in words))
print(words[:5])

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia']


In [77]:
# Build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)
print(f'itos: {itos}')

itos: {1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [ ]:
# Build a dataset of (block_size context) -> (next char) pairs.
# block_size bumped from 3 (previous notebooks) to 8 to give the wavenet hierarchy
# enough timesteps to compress through the tree (8 -> 4 -> 2 -> 1).
block_size = 8

def build_dataset(words):
    X, Y = [], []
    
    for word in words:
        context = [0] * block_size

        for c in word + '.':
            ix = stoi[c]
            X.append(context)
            Y.append(ix)

            context = context[1:] + [ix] # Crop and append

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(f'{X.shape=}, {Y.shape=}')
    return X, Y

In [79]:
# Create the trainings splits
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

for x, y in zip(Xtr[:20], Ytr[:20]):
    print(''.join(itos[ix.item()] for ix in x), '=>', itos[y.item()])

X.shape=torch.Size([182625, 8]), Y.shape=torch.Size([182625])
X.shape=torch.Size([22655, 8]), Y.shape=torch.Size([22655])
X.shape=torch.Size([22866, 8]), Y.shape=torch.Size([22866])
........ => y
.......y => u
......yu => h
.....yuh => e
....yuhe => n
...yuhen => g
..yuheng => .
........ => d
.......d => i
......di => o
.....dio => n
....dion => d
...diond => r
..diondr => e
.diondre => .
........ => x
.......x => a
......xa => v
.....xav => i
....xavi => e


In [ ]:
# PyTorch-style layer modules. Each class follows the same pattern:
#   __init__: store parameters as attributes (self.weight, self.bias, etc.)
#   __call__: run forward pass; store output in self.out for downstream inspection
#   parameters(): return list of tensors this layer owns (for gradient collection)
# This is essentially a hand-rolled, simplified version of torch.nn.Module.

class Linear:

    def __init__(self, fan_in, fan_out, bias=True):
        self.weight = torch.randn((fan_in, fan_out)) / fan_in ** 0.5
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self, x):
        self.out = x @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out
    
    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])

    
# BatchNorm1D now handles both 2D inputs (B, C) AND 3D inputs (B, T, C).
# For 2D: reduce over dim 0 (the batch). For 3D: reduce over dims (0, 1) (batch + time).
# Running stats still have shape (C,) since we always normalize per-feature.
class BatchNorm1D:
    
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True

        # Parameters (trained with backpropogation)
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

        # Buffers (trained with a running 'momentum update')
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self, x):

        # Calculate the forward pass
        if self.training:
            if x.ndim == 2:
                dim = 0
            elif x.ndim == 3:
                dim = (0, 1)
            xmean = x.mean(dim, keepdim=True)
            xvar = x.var(dim, keepdim=True, unbiased=True)
        else:
            xmean = self.running_mean
            xvar = self.running_var

        # Normalize to unit variance
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta
        
        # Update the buffers
        if self.training:
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar

        return self.out
    
    def parameters(self):
        return [self.gamma, self.beta]
    

class Tanh:
    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out
    
    def parameters(self):
        return []


class Embedding:
    def __init__(self, num_embeddings, embedding_dim):
        self.weight = torch.randn((num_embeddings, embedding_dim))

    def __call__(self, IX):
        self.out = self.weight[IX]
        return self.out

    def parameters(self):
        return [self.weight]


# FlattenConsecutive is the wavenet trick: given (B, T, C), reshape to (B, T//n, C*n).
# This "fuses" n consecutive timesteps into a single timestep with n times the channels.
# Repeatedly applying with n=2 builds a binary-tree hierarchy through the time dimension.
class FlattenConsecutive:
    def __init__(self, n):
        self.n = n

    def __call__(self, x):
        B, T, C = x.shape
        x = x.view(B, T // self.n, C * self.n)
        if x.shape[1] == 1:
            x = x.squeeze(1)
        self.out = x
        return self.out

    def parameters(self):
        return []    
    

# Sequential chains its layers in order. The forward pass walks the list calling each
# layer on the previous output; .parameters() flattens parameters across all sub-layers.
class Sequential:
    def __init__(self, layers):
        self.layers = layers

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        self.out = x
        return self.out

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [ ]:
# The wavenet model: hierarchical pairing of timesteps.
# Starting from block_size=8 chars (each n_embed-dim after embedding), each
# FlattenConsecutive(2) halves the time dimension while doubling channels:
#   (B, 8, n_embed) -> (B, 4, n_embed*2) -> (B, 2, n_hidden*2) -> (B, 1, n_hidden*2) -> logits
# Structurally similar to a 1D dilated convolutional network, which is what the
# original WaveNet paper used for audio generation.

n_embed = 24            # Dimensionality of the character embedding vectors
n_hidden = 128          # Number of neurons in the hidden layer of MLP
seed = 2147483647       # For reproducibility between runs
torch.manual_seed(42)   # Seed for Torch

print(f"Setup: block_size={block_size}, n_embed={n_embed}, n_hidden={n_hidden}")

model = Sequential([
    Embedding(vocab_size, n_embed), 
    FlattenConsecutive(2), Linear(n_embed * 2, n_hidden, bias=False), BatchNorm1D(n_hidden), Tanh(), 
    FlattenConsecutive(2), Linear(n_hidden * 2, n_hidden, bias=False), BatchNorm1D(n_hidden), Tanh(), 
    FlattenConsecutive(2), Linear(n_hidden * 2, n_hidden, bias=False), BatchNorm1D(n_hidden), Tanh(), 
    Linear(n_hidden, vocab_size)
])

parameters = model.parameters()
print(f'# of Parameters: {sum(parameter.nelement() for parameter in parameters)}')
for parameter in parameters:
    parameter.requires_grad = True

In [82]:
# Same optimization as last time
max_steps = 200000
batch_size = 32
lossi = []

for i in range(max_steps):
    # Minibatch Construct
    ix = torch.randint(0, Xtr.shape[0], (batch_size, ))
    Xb, Yb = Xtr[ix], Ytr[ix]

    # Forward Pass
    logits = model(Xb)
    loss = F.cross_entropy(logits, Yb)

    # Backward Pass
    for parameter in parameters:
        parameter.grad = None
    loss.backward()

    # Update
    lr = 0.1 if i < max_steps / 2 else 0.01
    for parameter in parameters:
        parameter.data += -lr * parameter.grad
    
    # Track stats
    if i % 10000 == 0:
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())

      0/ 200000: 3.6596
  10000/ 200000: 2.0551
  20000/ 200000: 2.0342
  30000/ 200000: 2.6254
  40000/ 200000: 2.1558
  50000/ 200000: 1.7977
  60000/ 200000: 2.2949
  70000/ 200000: 1.8409
  80000/ 200000: 1.6751
  90000/ 200000: 2.1980
 100000/ 200000: 1.9085
 110000/ 200000: 1.8829
 120000/ 200000: 1.6515
 130000/ 200000: 1.7502
 140000/ 200000: 1.7040
 150000/ 200000: 1.7704
 160000/ 200000: 1.8297
 170000/ 200000: 1.6398
 180000/ 200000: 1.5088
 190000/ 200000: 1.9149


In [ ]:
# Inspect per-layer output shapes + plot the smoothed loss curve.
# .out is populated by the training loop above, so this works (would error if
# we tried inspecting before any forward pass).
# The loss curve is smoothed by averaging over 1000-step buckets, otherwise per-step
# noise drowns out the trend.
for layer in model.layers:
    print(layer.__class__.__name__, ':', tuple(layer.out.shape))
plt.plot(torch.tensor(lossi).view(-1, 1000).mean(1))

In [ ]:
# Switch every BatchNorm1D layer from training to eval mode.
# In training mode, BN uses the per-batch mean/std. In eval mode, it uses the
# stored running_mean / running_var (the EMA tracked during training).
# CRITICAL: forgetting this is the BatchNorm-at-inference bug that produces
# garbage samples even though training loss looks fine.
for layer in model.layers:
    layer.training = False

In [85]:
# Evaluate the training Loss
@torch.no_grad() # Disabled gradient tracking
def split_loss(split):
    x,y = {
        'train': {Xtr, Ytr},
        'val': {Xdev, Ydev},
        'test': {Xte, Yte},
    }[split]
    logits = model(x)
    loss = F.cross_entropy(logits, y)
    print(split, loss.item())

split_loss('train')
split_loss('val')

train 1.7869818210601807
val 1.9925613403320312


In [86]:
# Sample from the model
gen = torch.Generator().manual_seed(seed + 10)

for _ in range(20):
    out = []
    context = [0] * block_size # Initialize with all
    
    while True:
        # Forward pass
        logits = model(torch.tensor([context]))
        probabilities = F.softmax(logits, dim=1)

        # Sample from the distribution
        ix = torch.multinomial(probabilities, num_samples=1, generator=gen).item()

        # Shift the context window and track the samples
        context = context[1:] + [ix]
        out.append(ix)

        # If we sample '.': break
        if ix == 0:
            break
    
    print(''.join(itos[i] for i in out)) # Decode and print the generated word

carmah.
amelia.
khyrir.
xithi.
khalani.
emmahne.
fares.
railah.
isara.
keagan.
ihvik.
legyn.
hamond.
niquint.
sulina.
livia.
corter.
gianryn.
kaileen.
rudi.
